# Perfilamiento de un conjunto de datos — calidad del agua

En este tutorial recorrerás el perfilamiento de un conjunto de datos real antes de modelarlo: 3.276 muestras de agua, nueve variables fisicoquímicas y una etiqueta de potabilidad. El perfilamiento es el paso que responde *qué tengo entre manos* —cuánto falta, qué está fuera de rango, qué tan desbalanceadas están las clases— y deja esas decisiones por escrito para que otra persona pueda repetirlas.

Específicamente, verás cómo realizar los siguientes procesos:

1. Importar las librerías requeridas y fijar los parámetros del análisis.
2. Cargar el conjunto de datos y revisar su dimensión y tipos.
3. Cuantificar los valores ausentes y los registros duplicados.
4. Obtener los estadísticos descriptivos y el balance de clases.
5. Contrastar los valores contra rangos de referencia usando máscaras booleanas de NumPy.
6. Detectar valores atípicos con el puntaje z.
7. Construir el perfil por clase con Pandas y repetir la misma consulta con Polars en modo diferido.

> Este cuaderno acompaña la Unidad 1 del NEEXO (*Reproducibilidad, entornos y manipulación de datos*). Las cifras de las observaciones corresponden a la ejecución de este mismo cuaderno sobre el archivo incluido en `data/`, con Python 3.13, pandas 3.0.5, NumPy 2.5.3 y Polars 1.44.2.

## 1. Importación de librerías y parámetros del análisis

Usaremos tres librerías, cada una en el papel que mejor desempeña:

* `pandas`: carga el archivo y entrega la tabla con nombres de columna, tipos y operaciones de agrupación.
* `numpy`: opera sobre los valores como arreglos, sin bucles, cuando solo nos interesa el cálculo numérico.
* `polars`: repite la misma consulta en modo diferido para comparar el diseño de las dos librerías.

De ellas usaremos en particular:

* `read_csv()`: función para cargar el archivo delimitado por comas.
* `isna()`: método que marca con `True` cada valor ausente.
* `duplicated()`: método que marca las filas repetidas.
* `describe()`: método que resume cada variable numérica.
* `groupby()`: método para agrupar filas y agregar por clase.
* `np.isnan()`, `np.nanmean()`, `np.nanstd()`: funciones de NumPy que tratan explícitamente los valores ausentes.
* `pl.LazyFrame`: estructura de Polars que construye un plan de consulta antes de ejecutarlo.

In [1]:
import numpy as np
import pandas as pd
import polars as pl

Antes de tocar los datos, fijamos los dos parámetros del análisis en un solo lugar: la ruta del archivo y los rangos de referencia.

Escribirlos como constantes al inicio no es un detalle de estilo. La ruta es **relativa al repositorio**, de modo que el cuaderno corre igual en cualquier computador donde esté clonado el proyecto; y los límites quedan a la vista, con su fuente, en vez de aparecer como números sueltos dentro de una comparación más adelante.

In [2]:
RUTA = "data/calidad_agua.csv"

# Rangos de referencia tomados del diccionario de datos del conjunto
LIMITES = {"pH": (6.5, 8.5),
           "Cloraminas": (0, 4),
           "Trihalometanos": (0, 80),
           "Turbidez": (0, 5)}

## 2. Carga de datos

Realizaremos la carga con la función `read_csv()` de Pandas, indicando la ruta del archivo:

In [3]:
df = pd.read_csv(RUTA)

Lo primero que se le pregunta a una tabla recién cargada es su tamaño. El atributo `shape` devuelve la tupla *(filas, columnas)*:

In [4]:
df.shape

(3276, 10)

El resultado es `(3276, 10)`: 3.276 muestras y 10 columnas, es decir, las nueve variables fisicoquímicas más la etiqueta `Potabilidad`.

Conviene anotar esta cifra, porque es la que usaremos como guarda del cuaderno: si mañana el archivo cambia de tamaño sin que nadie avise, el análisis debe fallar de inmediato en lugar de producir un resultado distinto en silencio.

Ahora revisaremos los tipos de dato. El método `dtypes` devuelve el tipo de cada columna y `value_counts()` los agrupa para verlos de un vistazo:

In [5]:
df.dtypes.value_counts()

float64    9
int64      1
Name: count, dtype: int64

Todas las columnas son numéricas: nueve variables en punto flotante y la etiqueta como entero. Es una buena noticia para este conjunto —no hay texto que codificar— pero también una advertencia: que `Potabilidad` sea un entero **no** significa que sea una variable numérica. Es una categoría disfrazada de número, y tratarla como continua (promediarla, por ejemplo) no tendría sentido.

## 3. Calidad del conjunto: valores ausentes y duplicados

### Valores ausentes

`isna()` devuelve una tabla de la misma forma con `True` donde falta el dato. Como en Python `True` vale 1, el promedio de esa tabla por columna es directamente la **proporción** de ausentes, que multiplicamos por 100 para leerla como porcentaje:

In [6]:
nulos = (df.isna().mean() * 100).round(1)
nulos[nulos > 0]

pH                15.0
Sulfatos          23.8
Trihalometanos     4.9
dtype: float64

Tres variables tienen datos ausentes: el pH con un 15,0 %, los sulfatos con un 23,8 % y los trihalometanos con un 4,9 %.

La magnitud importa más de lo que parece. Eliminar las filas incompletas con `dropna()` parece la salida rápida, pero en este conjunto dejaría 2.011 de las 3.276 filas: se perdería el 38,6 % de las muestras, y de forma no aleatoria: si los sulfatos faltan con más frecuencia en cierto tipo de fuente de agua, el conjunto que queda ya no representa a la población original. Por eso el perfilamiento **cuantifica** los ausentes y deja la decisión de qué hacer con ellos —imputar, descartar o modelar la ausencia— documentada y para más adelante.

### Registros duplicados

`duplicated()` marca cada fila que ya apareció antes, y `sum()` las cuenta:

In [7]:
int(df.duplicated().sum())

0

No hay filas repetidas. Si las hubiera, la pregunta antes de borrarlas sería si se trata de un error de carga o de dos muestras distintas que casualmente coinciden en las nueve mediciones: en un conjunto con variables continuas, lo segundo es muy improbable y casi siempre indica una duplicación técnica.

## 4. Estadísticos descriptivos y balance de clases

`describe()` resume cada variable numérica. Lo transponemos con `.T` para que cada variable sea una fila —más cómodo de leer con diez columnas— y nos quedamos con las cinco medidas que interesan en un perfilamiento:

In [8]:
df.describe().T[["count", "mean", "std", "min", "max"]].round(2)

,count,mean,std,min,max
pH,2785.0,7.08,1.59,0.00,14.00
Dureza,3276.0,196.37,32.88,47.43,323.12
Sólidos,3276.0,22014.09,8768.57,320.94,61227.20
Cloraminas,3276.0,7.12,1.58,0.35,13.13
Sulfatos,2495.0,333.78,41.42,129.00,481.03
Conductividad,3276.0,426.21,80.82,181.48,753.34
Carbono_orgánico,3276.0,14.28,3.31,2.20,28.30
Trihalometanos,3114.0,66.40,16.18,0.74,124.00
Turbidez,3276.0,3.97,0.78,1.45,6.74
Potabilidad,3276.0,0.39,0.49,0.00,1.00


Hay tres cosas que leer en esta tabla, y la primera suele pasarse por alto:

* La columna **`count` no es igual en todas las filas**. `describe()` cuenta valores no nulos, así que esa columna ya está mostrando los ausentes de la sección anterior desde otro ángulo.
* Los **órdenes de magnitud son muy distintos**: los sólidos se miden en decenas de miles y el pH en unidades. Cualquier método basado en distancias exigiría estandarizar antes.
* **`min` y `max`** delatan valores imposibles o extremos que los rangos de referencia confirmarán en la sección 5.

Una advertencia técnica sobre `std`: Pandas calcula la desviación **muestral** (`ddof=1`) y NumPy, por defecto, la **poblacional** (`ddof=0`). Sobre la dureza, la diferencia es de 32,88 frente a 32,87. A dos decimales parece irrelevante, pero si se estandariza con una y se filtran atípicos con la otra, las muestras que caen justo en el borde cambian de lado.

### Balance de clases

En un problema de clasificación, saber cómo se reparte la etiqueta es parte del perfilamiento. El parámetro `normalize=True` de `value_counts()` devuelve proporciones en lugar de conteos:

In [9]:
df["Potabilidad"].value_counts(normalize=True).round(3)

Potabilidad
0    0.61
1    0.39
Name: proportion, dtype: float64

El reparto es de 61 % no potable y 39 % potable. Es un desbalance moderado: no exige técnicas de remuestreo, pero sí fija el punto de comparación. Un clasificador que respondiera «no potable» siempre acertaría el 61 % de las veces, de modo que **la exactitud por debajo de 0,61 no significa nada** y conviene acompañarla de sensibilidad y precisión por clase.

## 5. Rangos de referencia con máscaras booleanas de NumPy

Hasta aquí describimos la distribución. Ahora contrastamos los valores contra el criterio externo: los rangos del diccionario de datos.

La herramienta es la **máscara booleana**. Al comparar un arreglo de NumPy con un número, la operación se aplica a todos los elementos a la vez y devuelve un arreglo de `True`/`False` del mismo tamaño; sumarlo cuenta los casos. Es el reemplazo vectorizado del bucle `for`, y se lee como la condición que expresa:

In [10]:
print("Valores fuera del rango de referencia:")
for col, (lo, hi) in LIMITES.items():
    v = df[col].to_numpy()
    fuera = int(((v < lo) | (v > hi)).sum())     # cada NaN compara como False
    medidos = int((~np.isnan(v)).sum())
    print(f"  {col:<15} {fuera:>5} de {medidos} medidos")

Valores fuera del rango de referencia:
  pH               1457 de 2785 medidos
  Cloraminas       3187 de 3276 medidos
  Trihalometanos    602 de 3114 medidos
  Turbidez          314 de 3276 medidos


Los resultados son contundentes: 1.457 muestras fuera de rango en pH (de 2.785 medidas), 3.187 de 3.276 en cloraminas, 602 de 3.114 en trihalometanos y 314 de 3.276 en turbidez.

Que casi la totalidad de las muestras incumpla el rango de cloraminas no se interpreta como un hallazgo sanitario, sino como una señal de que el rango de referencia del diccionario y la unidad en que vienen los datos pueden no corresponder. Un perfilamiento serio levanta esa pregunta antes de modelar.

Un segundo detalle, más fino: estas cifras son sensibles al borde. Once valores de pH y de turbidez caen a menos de cinco milésimas del límite. Si las mediciones se redondearan a dos decimales antes de comparar —que es como las reporta un laboratorio— los conteos bajarían a 1.454 y 311. La conclusión no cambia, pero la cifra exacta sí: cuando se reporta un conteo contra un umbral, hay que decir también con qué precisión se comparó.

Y aquí está el detalle que da nombre al comentario del código. **Toda comparación con `NaN` devuelve `False`**, incluida `NaN == NaN`. Los 491 valores ausentes de pH no se contaron como fuera de rango… pero tampoco como dentro: desaparecieron de la cuenta sin dejar rastro. Por eso la línea calcula aparte el denominador con `~np.isnan(v)`: sin él, el lector supondría que los 1.454 son sobre 3.276 y estaría leyendo mal la cifra.

## 6. Detección de valores atípicos con el puntaje z

El puntaje z expresa cada valor como el número de desviaciones estándar que lo separan de la media, de modo que `|z| > 3` marca los valores inusualmente alejados del centro.

Usamos `np.nanmean()` y `np.nanstd()`, las variantes que ignoran los ausentes: con `mean()` y `std()` a secas, un solo `NaN` contaminaría el resultado y devolvería `NaN` para todo el arreglo.

In [11]:
x = df["Dureza"].to_numpy()
z = (x - np.nanmean(x)) / np.nanstd(x)
int((np.abs(z) > 3).sum())

18

La dureza tiene 18 valores atípicos, el 0,5 % de las muestras.

Dos precisiones sobre el método, porque el umbral de 3 se usa con demasiada ligereza. Primero, el puntaje z supone una distribución aproximadamente simétrica: sobre una variable muy sesgada marcaría como atípica toda la cola larga, que puede ser perfectamente legítima. Segundo, la media y la desviación se calculan **con los atípicos dentro**, así que los valores extremos inflan la desviación y encubren a sus vecinos. Cuando esto preocupa, el rango intercuartílico es la alternativa robusta, porque la mediana y los cuartiles no se mueven con los extremos.

Nótese además que `np.nanstd()` usa `ddof=0`: es la misma diferencia poblacional/muestral de la sección 4, y aquí sí desplaza el umbral.

## 7. Perfil por clase con Pandas

El último paso del perfilamiento es comparar las variables entre las dos clases: si alguna se comporta de forma distinta según la potabilidad, es candidata a ser informativa para un modelo posterior.

`groupby()` divide las filas por el valor de `Potabilidad` y `mean()` promedia las columnas seleccionadas dentro de cada grupo:

In [12]:
(df.groupby("Potabilidad")[["pH", "Dureza", "Sólidos", "Turbidez"]]
   .mean()
   .round(2))

,pH,Dureza,Sólidos,Turbidez
Potabilidad,,,,
0,7.09,196.73,21777.49,3.97
1,7.07,195.80,22383.99,3.97


Las medias de las dos clases se parecen mucho. Eso también es un resultado del perfilamiento, y uno útil: anticipa que ninguna de estas cuatro variables separa las clases **por sí sola**, y que un modelo lineal simple difícilmente funcionará. La separación, si existe, estará en combinaciones de variables, que es justamente lo que un modelo no lineal puede capturar.

Recuerde que `mean()` ignora los ausentes: el promedio de pH de cada grupo se calcula sobre las muestras que sí tienen pH, no sobre todas las del grupo.

## 8. La misma consulta con Polars en modo diferido

Repetiremos una consulta equivalente con Polars para contrastar el diseño de las dos librerías.

Pandas es de **ejecución inmediata**: cada operación se calcula en el momento en que se escribe. Polars ofrece además la **ejecución diferida**: `lazy()` no calcula nada, sino que acumula las operaciones en un plan de consulta que un optimizador reescribe antes de ejecutarlo. La consulta solo corre cuando se llama a `collect()`.

In [13]:
consulta = (pl.read_csv(RUTA).lazy()
            .filter(pl.col("Sólidos") > 20000)
            .group_by("Potabilidad")
            .agg(pl.col("pH").mean().round(2).alias("pH_medio"),
                 pl.len().alias("n_filas"))
            .sort("Potabilidad"))

La celda anterior no leyó ni una fila: `consulta` es solo el plan. Podemos inspeccionarlo antes de ejecutarlo con `explain()`, que es la forma de ver qué decidió el optimizador:

In [14]:
print(consulta.explain())

SORT BY [col("Potabilidad")]
  AGGREGATE[maintain_order: false]
    [col("pH").mean().round().alias("pH_medio"), len().alias("n_filas")] BY [col("Potabilidad")]
    FROM
    simple π 2/2 ["Potabilidad", "pH"]
      FILTER col("Sólidos") > 20000.0
      FROM
        DF ["pH", "Dureza", "Sólidos", "Cloraminas", ...]; PROJECT["Potabilidad", "pH", "Sólidos"] 3/10 COLUMNS


En el plan aparece `PROJECT 3/10 COLUMNS`: aunque el archivo tiene diez columnas, el optimizador detectó que la consulta solo necesita `Potabilidad`, `pH` y `Sólidos`, y descartó las otras siete antes de leerlas. También empujó el filtro hacia la lectura, de modo que las filas con menos de 20.000 de sólidos no llegan a materializarse.

Esa es la ventaja real del modo diferido: no es que las operaciones sean más rápidas una por una, sino que al conocer la consulta completa el optimizador puede **hacer menos trabajo**. Por eso la ganancia aparece en consultas con filtros y selección de columnas, y se desvanece cuando no hay nada que podar.

Ahora sí ejecutamos el plan con `collect()`:

In [15]:
consulta.collect()

Potabilidad,pH_medio,n_filas
i64,f64,u32
0,6.84,1082
1,7.16,707


El resultado: entre las muestras con más de 20.000 de sólidos, el pH medio es 6,84 en las no potables (1.082 filas) y 7,16 en las potables (707 filas).

Vale la pena detenerse en la columna `n_filas`, porque esconde la trampa más frecuente al migrar entre las dos librerías. `pl.len()` cuenta **las filas del grupo**, tengan pH o no. El equivalente aparente en Pandas, `.agg("count")`, cuenta **los valores no nulos de la columna**, y sobre este mismo subconjunto devolvería 902 y 607: una diferencia de 280 registros, que son exactamente las filas con pH ausente.

Ninguna de las dos es incorrecta; son dos preguntas distintas que se escriben casi igual. De ahí el nombre explícito de la columna: `n_filas` dice lo que cuenta, y `n` no habría dicho nada. Si lo que se quiere es el número de mediciones de pH, la expresión en Polars es `pl.col("pH").count()`.

## Cierre

En este tutorial perfilamos un conjunto de datos real antes de modelarlo. Revisamos su dimensión y sus tipos, cuantificamos los valores ausentes y los duplicados, leímos los estadísticos descriptivos y el balance de clases, contrastamos los valores contra rangos de referencia con máscaras booleanas de NumPy, detectamos atípicos con el puntaje z y construimos el perfil por clase, primero con Pandas y después con Polars en modo diferido.

Tres ideas para llevarse, más allá de los comandos:

* El perfilamiento no limpia los datos: los **caracteriza** y documenta las decisiones que vendrán después.
* Los valores ausentes no son neutros. Desaparecen en silencio de las comparaciones (`NaN` compara siempre `False`) y de los conteos, y cambian el denominador de cualquier cifra que se reporte.
* Funciones con el mismo nombre en librerías distintas pueden responder preguntas distintas. Nombrar las columnas por lo que realmente cuentan evita la mayor parte de esas discusiones.

---

Documentación de `read_csv()` en Pandas: [enlace](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)

Sobre indexación booleana en NumPy puedes consultar [este enlace](https://numpy.org/doc/stable/user/basics.indexing.html)

Sobre la ejecución diferida en Polars puedes consultar el [sitio web oficial](https://docs.pola.rs/user-guide/lazy/)

---
*Creado por: María Fernanda Ochoa*
*Asignatura: Programación avanzada — Unidad 1*
*Versión de: Noviembre 7, 2026*
*Escuela Colombiana de Ingeniería Julio Garavito*